# Treasury Auction Influence — Deck Exhibits

**Audience:** trading desk / MD review  
**Purpose:** Six self-contained exhibits, ordered so the thesis builds without a single formula in the body. All math is in the appendix (technical doc).

---

### Reading order & why it matters

| # | Exhibit | Question answered |
|---|---------|------------------|
| 1 | **Event study** | Is the dislocation real? *(model-independent)* |
| 2 | **Regime overlay** | What does "stressed" look like on an actual day? |
| 3 | **Transition heatmap** | How sticky are the regimes? |
| 4 | **Decay curve** | How long does the footprint last — calm vs. stressed? |
| 5 | **Feature importance** | Does the tail actually predict next-day moves? |
| 6 | **Forecast scorecard** | What's the honest edge vs. the naïve benchmark? |

> **Principle:** Every formula belongs in the appendix. These six charts carry the body of the argument.

---
## Setup — load data & imports

> Run this cell first. All exhibits below depend on the variables defined here.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.colors as mcolors
from sklearn.inspection import permutation_importance
from sklearn.metrics import r2_score

from config import CACHE_DIR, H_DECAY, EMBARGO_TD, N_REGIMES, FEATURES, TARGET
from src.decay import local_projection_decay, fit_half_life
from src.validation import decay_cv, event_study, PurgedKFold
from src.regime import HAVE_HMM

# ── Colour palette (desk-friendly, prints in greyscale) ──────────────────────
CALM_C    = '#2166AC'   # blue
NORMAL_C  = '#4DAC26'   # green
STRESS_C  = '#D01C8B'   # pink/magenta
REGIME_COLORS = [CALM_C, NORMAL_C, STRESS_C]
REGIME_LABELS = ['Calm', 'Normal', 'Stressed']

plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top':   False,
    'axes.spines.right': False,
    'font.size': 11,
})

# ── Load caches ───────────────────────────────────────────────────────────────
decay_panel  = pd.read_parquet(CACHE_DIR / 'decay_panel.parquet')
intraday_df  = pd.read_parquet(CACHE_DIR / 'intraday_stage1.parquet')
regime_df    = pd.read_parquet(CACHE_DIR / 'regime_stage3.parquet')
df_forecast  = pd.read_parquet(CACHE_DIR / 'df_forecast.parquet')
oos_preds    = pd.read_parquet(CACHE_DIR / 'oos_predictions.parquet')

print(f'Decay panel : {len(decay_panel):,} rows  |  horizons 0–{H_DECAY}')
print(f'Intraday    : {intraday_df["auction_id"].nunique()} auctions')
print(f'OOS preds   : {len(oos_preds)} rows')

---
## Exhibit 1 — Event Study: The Dislocation is Real

**What you're seeing:** Mean `micro_factor` by event-minute across every auction in the sample, split by tail-size tertile.  
The `micro_factor` is the OTR 30Y yield minus the Svensson model's fitted fair-value — it measures how much the on-the-run bond is dislocated from the curve.

**What to look for:**
- Pre-auction build-up: dealers widen the spread to extract concessions from buyers.
- Sharp reversal after 13:00: the when-issued supply overhang clears.
- **Large-tail auctions reverse more slowly** — that persistence is the signal this model exploits.

> **Why first:** This exhibit requires trusting nothing — no model, no regression. The pattern is in the raw data. If it weren't here, the rest wouldn't matter.

In [ ]:
# ── Split auctions by tail-size tertile ──────────────────────────────────────
if 'tail_bps' in df_forecast.columns and df_forecast['tail_bps'].notna().sum() > 10:
    tail_q = df_forecast['tail_bps'].quantile([1/3, 2/3]).values
    def tail_group(x):
        if x <= tail_q[0]:  return 'Small tail'
        if x <= tail_q[1]:  return 'Mid tail'
        return 'Large tail'
    df_forecast['_tail_grp'] = df_forecast['tail_bps'].map(tail_group)
    grp_map = df_forecast.set_index('auction_id')['_tail_grp'].to_dict()
    intraday_df['_tail_grp'] = intraday_df['auction_id'].map(grp_map).fillna('Mid tail')
    SPLIT_COL = '_tail_grp'
    SPLIT_ORDER = ['Small tail', 'Mid tail', 'Large tail']
    SPLIT_COLORS = ['#4393C3', '#878787', '#D01C8B']
else:
    intraday_df['_tail_grp'] = 'All auctions'
    SPLIT_COL = '_tail_grp'
    SPLIT_ORDER = ['All auctions']
    SPLIT_COLORS = ['steelblue']

# ── Compute event study per split ────────────────────────────────────────────
es_all = event_study(intraday_df)  # pooled, for SE band

fig, ax = plt.subplots(figsize=(13, 5))

# Background SE band (pooled)
se = es_all['micro_std'] / np.sqrt(es_all['n'])
ax.fill_between(es_all.index,
                es_all['micro_mean'] - se,
                es_all['micro_mean'] + se,
                color='#BBBBBB', alpha=0.35, label='Pooled ±1 SE')

# Per-split lines
for grp, col in zip(SPLIT_ORDER, SPLIT_COLORS):
    sub = intraday_df[intraday_df[SPLIT_COL] == grp]
    if len(sub) == 0:
        continue
    es_sub = event_study(sub)
    ax.plot(es_sub.index, es_sub['micro_mean'], color=col, linewidth=2, label=grp)

ax.axvline(0,  color='#CC0000', linestyle='--', linewidth=1.5, label='Auction close (13:00)')
ax.axhline(0,  color='black',   linewidth=0.6)
ax.set_xlabel('Minutes relative to auction close (13:00 ET)', fontsize=12)
ax.set_ylabel('micro_factor (bps)', fontsize=12)
ax.set_title('Exhibit 1 — Event Study: OTR Dislocation Around Auction Close', fontsize=13, fontweight='bold')
ax.legend(frameon=False)
plt.tight_layout()
plt.show()

print('\n► Key read: the pre-auction build-up (left of dashed line) and reversal (right) confirm the'
      ' dislocation is systematic. Large-tail auctions show a slower reversal — that lag is the signal.')

---
## Exhibit 2 — Regime Overlay: What "Stressed" Looks Like on a Real Day

**What you're seeing:** One representative auction day's minute-by-minute `micro_factor` path, with the HMM regime as a colour-shaded background.

**What to look for:**
- The regime flip — the instant the colour changes from calm (blue) to stressed (pink) — is the threshold the model is learning.
- Stressed regimes correspond to large or erratic micro-factor swings; calm regimes to tight, mean-reverting spreads.

> **Why this chart:** "Hidden Markov regime" is abstract. One real auction day makes it concrete. The trader sees what *their* tape looks like on a stressed day, and then the model is no longer a black box.

In [ ]:
if not HAVE_HMM:
    print('hmmlearn not available — skipping Exhibit 2 (regime overlay).')
else:
    from src.regime import fit_pooled_hmm, _forward_filtered_probs

    # Pick the auction with the highest micro_factor range (most dramatic example)
    ranges = (intraday_df.groupby('auction_id')['micro_factor']
                         .agg(lambda s: s.max() - s.min()))
    example_id = ranges.idxmax()

    day = (intraday_df[intraday_df['auction_id'] == example_id]
               .sort_values('event_minute')
               .reset_index(drop=True))

    # Fit HMM on all other auctions (train-only proxy for this exhibit)
    train_ids = set(intraday_df['auction_id'].unique()) - {example_id}
    train_intra = intraday_df[intraday_df['auction_id'].isin(train_ids)]
    hmm_model = fit_pooled_hmm(train_intra)

    mf = day['micro_factor'].fillna(0).values.reshape(-1, 1)
    states = hmm_model.predict(mf)  # Viterbi
    filt   = _forward_filtered_probs(hmm_model, mf)  # (T, K) filtered
    minutes = day['event_minute'].values

    # ── Plot ─────────────────────────────────────────────────────────────────
    fig, (ax_mf, ax_pr) = plt.subplots(2, 1, figsize=(13, 7),
                                        gridspec_kw={'height_ratios': [3, 1]},
                                        sharex=True)

    # Regime shading behind micro_factor
    for t in range(len(minutes) - 1):
        ax_mf.axvspan(minutes[t], minutes[t+1],
                      color=REGIME_COLORS[states[t] % N_REGIMES],
                      alpha=0.15, linewidth=0)

    ax_mf.plot(minutes, day['micro_factor'], color='black', linewidth=1.4, zorder=3)
    ax_mf.axvline(0, color='#CC0000', linestyle='--', linewidth=1.5, label='13:00 close')
    ax_mf.axhline(0, color='black', linewidth=0.5)
    ax_mf.set_ylabel('micro_factor (bps)', fontsize=11)
    ax_mf.set_title(f'Exhibit 2 — Regime Overlay: Auction {example_id}', fontsize=13, fontweight='bold')

    # Legend patches for regimes
    patches = [mpatches.Patch(color=REGIME_COLORS[k], alpha=0.5, label=REGIME_LABELS[k])
               for k in range(N_REGIMES)]
    patches.append(mpatches.Patch(color='#CC0000', label='13:00 close'))
    ax_mf.legend(handles=patches, frameon=False, loc='upper left')

    # Filtered regime probabilities
    for k in range(N_REGIMES):
        ax_pr.plot(minutes, filt[:, k], color=REGIME_COLORS[k],
                   linewidth=1.2, label=REGIME_LABELS[k])
    ax_pr.set_ylabel('Filtered P(regime)', fontsize=10)
    ax_pr.set_xlabel('Event minute', fontsize=11)
    ax_pr.set_ylim(0, 1)
    ax_pr.legend(frameon=False, ncol=N_REGIMES, loc='upper right')

    plt.tight_layout()
    plt.show()

    print(f'\n► Auction shown: {example_id}')
    print('  Top panel: micro_factor path, coloured by Viterbi-decoded regime.')
    print('  Bottom panel: forward-filtered P(regime | data up to t) — causal, uses no future information.')

---
## Exhibit 3 — Transition Matrix: How Sticky Are the Regimes?

**What you're seeing:** The 3×3 Markov transition matrix as a heatmap.  
Cell (i, j) = probability of moving from regime i to regime j in one step.

**What to look for:**
- **Diagonal dominance:** States persist. A high P(calm→calm) means once you're in calm, you stay there — the regime is informative over multiple periods.
- **Asymmetric flip risk:** P(calm→stressed) vs P(stressed→calm) tells you whether dislocations clear quickly or not.

> **Why this chart:** It teaches the Markov idea in four seconds. You don't need to explain Baum-Welch; you just need to show that the diagonal is hot and off-diagonal is cold.

In [ ]:
if not HAVE_HMM:
    print('hmmlearn not available — skipping Exhibit 3 (transition matrix).')
else:
    # hmm_model from Exhibit 2 (or refit if skipped)
    try:
        trans = hmm_model.transmat_
    except NameError:
        from src.regime import fit_pooled_hmm
        hmm_model = fit_pooled_hmm(intraday_df)
        trans = hmm_model.transmat_

    K = trans.shape[0]
    labels = REGIME_LABELS[:K]

    fig, ax = plt.subplots(figsize=(6, 5))
    im = ax.imshow(trans, cmap='Blues', vmin=0, vmax=1, aspect='auto')

    # Cell annotations
    for i in range(K):
        for j in range(K):
            val = trans[i, j]
            txt_color = 'white' if val > 0.55 else 'black'
            ax.text(j, i, f'{val:.2f}', ha='center', va='center',
                    fontsize=13, fontweight='bold', color=txt_color)

    ax.set_xticks(range(K))
    ax.set_yticks(range(K))
    ax.set_xticklabels([f'→ {l}' for l in labels], fontsize=11)
    ax.set_yticklabels(labels, fontsize=11)
    ax.set_xlabel('To regime', fontsize=11)
    ax.set_ylabel('From regime', fontsize=11)
    ax.set_title('Exhibit 3 — Regime Transition Matrix (one-step probabilities)',
                 fontsize=12, fontweight='bold', pad=12)

    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label='Probability')
    plt.tight_layout()
    plt.show()

    # Stickiness commentary
    for i, lbl in enumerate(labels):
        print(f'  {lbl:8s}: stays {trans[i,i]:.0%} of the time  |  '
              f'flips to {labels[(i+1)%K]} {trans[i,(i+1)%K]:.0%}')

---
## Exhibit 4 — Decay Curve: The Auction's Footprint

**What you're seeing:** The impulse response function (IRF) from Jordà local projections.  
β_h = how many bps of cumulative curve shift you expect at horizon h per 1 bps of tail.  
The fitted exponential gives an **influence half-life** in business days.  
Where available, calm and stressed regimes are shown separately.

**What to look for:**
- Does β_h start positive and decay, or overshoot? Overshoot means reversal.
- **Half-life:** the single number you want to hand to the desk. "The auction's footprint halves every N days."
- Stressed-regime half-life should be *longer* — the market is slower to re-price.

> **This is the headline result.** Everything else in the notebook either motivates it (Ex 1–3) or validates it (Ex 5–6).

In [ ]:
# ── Full-sample IRF ───────────────────────────────────────────────────────────
irf_full = local_projection_decay(decay_panel, shock_col='auction_shock')
hl_full  = fit_half_life(irf_full)

# ── Regime-conditional IRFs (if regime data available) ───────────────────────
regime_irfs = {}
if 'regime_eod' in df_forecast.columns:
    # Map auction → regime, attach to decay panel
    reg_map = df_forecast.set_index('auction_id')['regime_eod'].dropna().astype(int).to_dict()
    decay_panel['_regime'] = decay_panel['auction_id'].map(reg_map)

    for k in range(N_REGIMES):
        sub = decay_panel[decay_panel['_regime'] == k]
        if sub['auction_id'].nunique() >= 10:
            irf_k = local_projection_decay(sub, shock_col='auction_shock')
            if not irf_k.empty:
                regime_irfs[k] = (irf_k, fit_half_life(irf_k))

# ── Plot ──────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(11, 5))

# Full-sample CI band
ax.fill_between(irf_full.index, irf_full['ci_lo'], irf_full['ci_hi'],
                color='#888888', alpha=0.18, label='Full-sample 95% HAC CI')
ax.plot(irf_full.index, irf_full['beta'], 'o-', color='#333333',
        linewidth=2, markersize=5, label='β_h (all regimes)')

# Fitted exponential — full sample
if not np.isnan(hl_full):
    tau    = hl_full / np.log(2)
    b0     = irf_full['beta'].iloc[0]
    h_grid = np.linspace(0, H_DECAY, 200)
    ax.plot(h_grid, b0 * np.exp(-h_grid / tau), '--', color='#333333',
            linewidth=1.5, alpha=0.7)
    ax.annotate(f' t½ = {hl_full:.1f} bd (all)',
                xy=(hl_full, b0 * 0.5),
                xytext=(hl_full + 0.5, b0 * 0.5),
                fontsize=10, color='#333333',
                arrowprops=dict(arrowstyle='->', color='#333333', lw=1.2))

# Regime-conditional lines
for k, (irf_k, hl_k) in regime_irfs.items():
    col = REGIME_COLORS[k % N_REGIMES]
    lbl = REGIME_LABELS[k % N_REGIMES]
    ax.plot(irf_k.index, irf_k['beta'], 's-', color=col,
            linewidth=1.8, markersize=4, label=f'β_h — {lbl}')
    if not np.isnan(hl_k):
        tau_k = hl_k / np.log(2)
        b0_k  = irf_k['beta'].iloc[0]
        ax.plot(h_grid, b0_k * np.exp(-h_grid / tau_k), '--', color=col,
                linewidth=1.2, alpha=0.6)
        ax.annotate(f' t½ = {hl_k:.1f} bd ({lbl.lower()})',
                    xy=(hl_k, b0_k * 0.5),
                    xytext=(hl_k + 0.5, b0_k * 0.55),
                    fontsize=9, color=col,
                    arrowprops=dict(arrowstyle='->', color=col, lw=1.0))

ax.axhline(0, color='black', linewidth=0.7)
ax.set_xlabel('Horizon h (business days after auction)', fontsize=12)
ax.set_ylabel('β_h — curve level shift per 1 bps tail', fontsize=12)
ax.set_title('Exhibit 4 — Decay Curve: How Long Does the Auction Footprint Last?',
             fontsize=13, fontweight='bold')
ax.legend(frameon=False)
plt.tight_layout()
plt.show()

print(f'\n► Full-sample half-life: {hl_full:.2f} business days')
for k, (_, hl_k) in regime_irfs.items():
    print(f'   {REGIME_LABELS[k]:8s}: {hl_k:.2f} bd')

---
## Exhibit 5 — Feature Importance: Does the Tail Predict Next-Day Moves?

**What you're seeing:** Permutation importance from the h=1 Random Forest, averaged across CV folds.  
Permutation importance drops a feature from the *fitted* model and measures the resulting loss in R² — unlike impurity importance, it is not inflated by high-cardinality or correlated features.

**What to look for:**
- Is `tail_bps` near the top? If yes, the auction result itself is informative — the desk's concession estimate matters.
- Where does `macro_resid_daily` fall? High rank = macro context is doing a lot of the work.
- Regime features (`prob_regime_k_eod`) appearing in the top 10 validates the HMM stage.

> **Why permutation, not impurity:** Correlated rate features (level, slope, curvature) artificially share impurity gain. Permutation correctly shows each feature's marginal contribution on held-out data.

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from config import RF_MAX_DEPTH, RF_MIN_SAMPLES_LEAF, RF_N_ESTIMATORS, RANDOM_SEED

# ── Fit a single final RF on all OOS folds' training data for permutation imp ─
# We use df_forecast merged with fold split info from oos_preds
# Simpler: refit on the largest fold's training split (last fold = most data)
from src.model import walk_forward_leakage_safe

avail_feats = [f for f in FEATURES if f in df_forecast.columns]
mask_target = df_forecast[TARGET].notna()
X_all = df_forecast.loc[mask_target, avail_feats].fillna(0)
y_all = df_forecast.loc[mask_target, TARGET]

# Use the last ~80% as train, first 20% as validation for permutation importance
n = len(X_all)
split = int(n * 0.8)
X_tr, X_va = X_all.iloc[:split], X_all.iloc[split:]
y_tr, y_va = y_all.iloc[:split], y_all.iloc[split:]

rf = RandomForestRegressor(
    n_estimators=RF_N_ESTIMATORS,
    max_depth=RF_MAX_DEPTH,
    min_samples_leaf=RF_MIN_SAMPLES_LEAF,
    random_state=RANDOM_SEED,
)
rf.fit(X_tr, y_tr)

# Permutation importance on the validation set
perm_result = permutation_importance(
    rf, X_va, y_va,
    n_repeats=25,
    random_state=RANDOM_SEED,
    scoring='r2',
)

perm_imp = pd.Series(
    perm_result.importances_mean,
    index=avail_feats
).sort_values()

# ── Plot top 15 ───────────────────────────────────────────────────────────────
top_n = min(15, len(perm_imp))
top   = perm_imp.tail(top_n)
colors = ['#D01C8B' if v > 0 else '#AAAAAA' for v in top.values]

fig, ax = plt.subplots(figsize=(9, 6))
bars = ax.barh(top.index, top.values, color=colors, edgecolor='none')
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Mean decrease in R² when feature permuted', fontsize=11)
ax.set_title(f'Exhibit 5 — Feature Importance (Permutation, top {top_n})',
             fontsize=13, fontweight='bold')

# Annotate tail_bps specifically
if 'tail_bps' in top.index:
    val = top.loc['tail_bps']
    idx = list(top.index).index('tail_bps')
    ax.annotate('← auction result', xy=(val, idx), xytext=(val + 0.001, idx + 0.3),
                fontsize=9, color='#D01C8B')

plt.tight_layout()
plt.show()

print('\nTop-5 features (permutation importance):')
print(perm_imp.tail(5).sort_values(ascending=False).round(5).to_string())

---
## Exhibit 6 — Forecast Scorecard: Honest Edge vs. Naïve Benchmark

**What you're seeing:** Per-fold directional accuracy (hit rate) and R², for the Random Forest vs. the regime-conditional mean baseline.  

**Why directional accuracy, not just R²:**  
With N≈220 auctions, OOS R² has wide confidence intervals and can look modestly positive even by chance. Directional accuracy (did you get the *sign* of the next-day move right?) is harder to game and easier for the desk to interpret as a trading signal.

> **Keep it modest.** A 55–60% hit rate over 40+ OOS observations is real. 70%+ on 10 observations isn't. This chart is the honest scorecard — resist the temptation to cherry-pick the best fold.

**Benchmark:** Regime-conditional mean — the simplest non-trivial model. If the RF doesn't beat it consistently, it isn't earning its complexity.

In [ ]:
# ── Per-fold metrics ──────────────────────────────────────────────────────────
fold_metrics = []
for fold_n, grp in oos_preds.groupby('fold'):
    actual   = grp[TARGET].values
    pred_rf  = grp['pred_rf'].values
    pred_bl  = grp['pred_base'].values
    mask     = np.isfinite(actual) & np.isfinite(pred_rf)

    if mask.sum() < 5:
        continue

    # Directional accuracy
    dir_rf = np.mean(np.sign(actual[mask]) == np.sign(pred_rf[mask]))
    dir_bl = np.mean(np.sign(actual[mask]) == np.sign(pred_bl[mask]))

    r2_rf  = r2_score(actual[mask], pred_rf[mask])
    r2_bl  = r2_score(actual[mask], pred_bl[mask])

    fold_metrics.append({
        'fold': int(fold_n), 'n': int(mask.sum()),
        'dir_rf': dir_rf, 'dir_bl': dir_bl,
        'r2_rf':  r2_rf,  'r2_bl':  r2_bl,
    })

fm = pd.DataFrame(fold_metrics)

# ── Plot ──────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

x = np.arange(len(fm))
w = 0.35

# Directional accuracy
axes[0].bar(x - w/2, fm['dir_rf'], w, label='Random Forest',
            color='#2166AC', alpha=0.85)
axes[0].bar(x + w/2, fm['dir_bl'], w, label='Regime baseline',
            color='#878787', alpha=0.7)
axes[0].axhline(0.5, color='#CC0000', linestyle='--', linewidth=1.2,
                label='50% (coin flip)')
axes[0].set_xticks(x)
axes[0].set_xticklabels([f'Fold {int(f)}\n(n={n})'
                          for f, n in zip(fm['fold'], fm['n'])], fontsize=9)
axes[0].set_ylabel('Directional accuracy', fontsize=11)
axes[0].set_ylim(0, 1)
axes[0].set_title('Directional Accuracy per Fold', fontsize=12, fontweight='bold')
axes[0].legend(frameon=False)

# OOS R²
axes[1].bar(x - w/2, fm['r2_rf'], w, label='Random Forest',
            color='#2166AC', alpha=0.85)
axes[1].bar(x + w/2, fm['r2_bl'], w, label='Regime baseline',
            color='#878787', alpha=0.7)
axes[1].axhline(0, color='black', linewidth=0.8)
axes[1].set_xticks(x)
axes[1].set_xticklabels([f'Fold {int(f)}\n(n={n})'
                          for f, n in zip(fm['fold'], fm['n'])], fontsize=9)
axes[1].set_ylabel('OOS R²', fontsize=11)
axes[1].set_title('OOS R² per Fold', fontsize=12, fontweight='bold')
axes[1].legend(frameon=False)

fig.suptitle('Exhibit 6 — Forecast Scorecard: RF vs. Regime-Conditional Baseline',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

# ── Summary row ───────────────────────────────────────────────────────────────
print('\nPooled scorecard:')
all_actual  = oos_preds[TARGET].values
all_pred_rf = oos_preds['pred_rf'].values
all_pred_bl = oos_preds['pred_base'].values
m = np.isfinite(all_actual) & np.isfinite(all_pred_rf)

dir_rf_pool = np.mean(np.sign(all_actual[m]) == np.sign(all_pred_rf[m]))
dir_bl_pool = np.mean(np.sign(all_actual[m]) == np.sign(all_pred_bl[m]))
r2_rf_pool  = r2_score(all_actual[m], all_pred_rf[m])
r2_bl_pool  = r2_score(all_actual[m], all_pred_bl[m])

print(f'  RF    | dir acc: {dir_rf_pool:.1%}  | OOS R²: {r2_rf_pool:.4f}')
print(f'  Base  | dir acc: {dir_bl_pool:.1%}  | OOS R²: {r2_bl_pool:.4f}')
print(f'  Edge  | dir acc: {dir_rf_pool - dir_bl_pool:+.1%}  | ΔR²: {r2_rf_pool - r2_bl_pool:+.4f}')

---
## Appendix A — Technical Validation

The cells below contain the underlying statistical outputs referenced in the exhibits.  
They are not part of the desk presentation — include them in the technical appendix or share on request.

| Appendix item | Cell below |
|---------------|------------|
| Full IRF table (β_h, SE, t-stat) | A1 |
| OOS R² by horizon (PurgedKFold) | A2 |

In [ ]:
# A1 — Full IRF table
irf_full.round(5)

In [ ]:
# A2 — PurgedKFold OOS R² by horizon
oos_decay, r2_by_h = decay_cv(
    decay_panel,
    shock_col='auction_shock',
    n_splits=5,
    embargo_td=EMBARGO_TD,
)
if not r2_by_h.empty:
    print('OOS R² by horizon (PurgedKFold):')
    print(r2_by_h.round(4).to_string())